In [1]:
import os
import csv
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from openpyxl import Workbook
from openpyxl.styles import PatternFill
from openpyxl.utils import get_column_letter

In [6]:
# -----------------------
# Configuration
# -----------------------
MODEL_PATH = "cnn_model_4.h5"
DATASET_ROOT = "./dataset_new"   # expects subfolders 'normal' and 'osteoporosis'
OUTPUT_CSV = "osteoporosis_predictions_cnn.csv"
OUTPUT_XLSX = "osteoporosis_predictions_cnn.xlsx"

# Terminal colors (ANSI)
GREEN = "\033[92m"
RED = "\033[91m"
YELLOW = "\033[93m"
RESET = "\033[0m"

# Excel fills
GREEN_FILL = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
RED_FILL = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
HEADER_FILL = PatternFill(start_color="BDD7EE", end_color="BDD7EE", fill_type="solid")

# Allowed image extensions
IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")

In [7]:
# -----------------------
# Helpers
# -----------------------
def infer_image_size_from_model(model):
    """Try to infer (height, width) from model.input_shape; fallback to (224,224)."""
    try:
        shape = model.input_shape
        # shape could be like (None, H, W, C) or (None, C, H, W)
        if isinstance(shape, list):
            shape = shape[0]
        # ignore batch dim
        if len(shape) == 4:
            _, a, b, c = shape
            # sometimes channels-first
            if a is None or b is None:
                return (224, 224)
            # handle case (None, channels, H, W)
            if a in (1, 3) and b not in (1, 3):
                # probably (None, C, H, W)
                return (b, c)
            else:
                return (a, b)
        elif len(shape) == 3:
            _, a, b = shape
            return (a, b)
    except Exception:
        pass
    return (224, 224)

def gather_image_paths(root_folder):
    """Recursively gather image paths and determine the actual label from parent folder name."""
    items = []
    for dirpath, _, filenames in os.walk(root_folder):
        # skip root folder files if any; consider folder name as actual class
        for fn in filenames:
            if fn.lower().endswith(IMG_EXTS):
                full = os.path.join(dirpath, fn)
                # true class is the immediate parent folder's basename
                true_class = os.path.basename(os.path.normpath(dirpath)).lower()
                items.append((full, true_class))
    return sorted(items)

def preprocess_image(path, target_size):
    img = load_img(path, target_size=target_size)
    arr = img_to_array(img).astype("float32") / 255.0
    # ensure channel last
    if arr.ndim == 3:
        pass
    else:
        # unexpected shape - try to reshape
        arr = np.resize(arr, (target_size[0], target_size[1], 3))
    arr = np.expand_dims(arr, axis=0)
    return arr

def decide_label_and_confidence(pred):
    """
    Accepts model prediction array and returns (predicted_label_str, confidence_float).
    Supports:
      - Sigmoid single output: shape (1,1) -> confidence = pred[0][0], label = osteoporosis if >= 0.5
      - Softmax two outputs: shape (1,2) -> confidence = prob of class 1, label accordingly
      - Other shapes: take last dim index of max
    Convention: class 0 -> 'normal', class 1 -> 'osteoporosis'
    """
    arr = np.array(pred)
    if arr.ndim == 0:
        # scalar
        conf = float(arr)
        label = "osteoporosis" if conf >= 0.3 else "normal"
        return label, conf
    # If shape (1,) or (1,1)
    if arr.size == 1:
        conf = float(arr.flatten()[0])
        label = "osteoporosis" if conf >= 0.3 else "normal"
        return label, conf
    # If two or more outputs
    # get probs for class 1 if exists
    flat = arr.reshape(-1, arr.shape[-1]) if arr.ndim > 1 else arr
    probs = flat[0]
    # if two-class softmax
    if probs.shape[0] == 2:
        conf = float(probs[1])
        label = "osteoporosis" if np.argmax(probs) == 1 else "normal"
        return label, conf
    # otherwise pick argmax as predicted class index
    idx = int(np.argmax(probs))
    # if we assume index 1 == osteoporosis
    conf = float(probs[idx])
    label = "osteoporosis" if idx == 1 else "normal"
    return label, conf

In [8]:
# -----------------------
# Main
# -----------------------
def main():
    # check dataset folders
    normal_dir = os.path.join(DATASET_ROOT, "normal")
    osteo_dir = os.path.join(DATASET_ROOT, "osteoporosis")
    if not os.path.isdir(normal_dir) and not os.path.isdir(osteo_dir):
        print(RED + f"Error: Neither '{normal_dir}' nor '{osteo_dir}' exist." + RESET)
        return

    # load model
    print(YELLOW + f"Loading model from '{MODEL_PATH}'..." + RESET)
    model = load_model(MODEL_PATH)
    img_h, img_w = infer_image_size_from_model(model)
    # Keras expects (height, width)
    target_size = (int(img_h), int(img_w))
    print(YELLOW + f"Using target image size: {target_size}" + RESET)

    # gather images
    images = []
    if os.path.isdir(normal_dir):
        images.extend(gather_image_paths(normal_dir))
    if os.path.isdir(osteo_dir):
        images.extend(gather_image_paths(osteo_dir))

    if len(images) == 0:
        print(RED + "No images found in the expected folders." + RESET)
        return

    # Prepare CSV and XLSX
    csv_rows = []
    header = ["Image Path", "Image Name", "Actual Class", "Predicted Class", "Confidence", "Status"]
    csv_rows.append(header)

    # Excel workbook
    wb = Workbook()
    ws = wb.active
    ws.title = "Predictions"
    ws.append(header)
    # style header
    for col_idx in range(1, len(header)+1):
        ws.cell(row=1, column=col_idx).fill = HEADER_FILL
        ws.column_dimensions[get_column_letter(col_idx)].width = 25

    # Process images
    total = len(images)
    correct_count = 0
    for i, (img_path, true_class_raw) in enumerate(images, start=1):
        true_class = "normal" if "normal" in true_class_raw.lower() else "osteoporosis"
        try:
            arr = preprocess_image(img_path, target_size)
            pred = model.predict(arr, verbose=0)
            pred_label, confidence = decide_label_and_confidence(pred)
        except Exception as e:
            pred_label = "error"
            confidence = 0.0
            print(RED + f"[{i}/{total}] ERROR processing {img_path}: {e}" + RESET)
            csv_rows.append([img_path, os.path.basename(img_path), true_class, pred_label, confidence, "Error"])
            ws.append([img_path, os.path.basename(img_path), true_class, pred_label, confidence, "Error"])
            # color last row red
            for c in range(1, 7):
                ws.cell(row=ws.max_row, column=c).fill = RED_FILL
            continue

        correct = (pred_label == true_class)
        status = "Correct" if correct else "Wrong"
        if correct:
            correct_count += 1

        # Terminal colored print
        color = GREEN if correct else RED
        print(color + f"[{i}/{total}] {os.path.basename(img_path)} => True: {true_class} | Pred: {pred_label} | Conf: {confidence:.4f} | {status}" + RESET)

        # Append CSV row
        csv_rows.append([img_path, os.path.basename(img_path), true_class, pred_label, f"{confidence:.6f}", status])

        # Append to Excel and color the status cell
        ws.append([img_path, os.path.basename(img_path), true_class, pred_label, float(confidence), status])
        row_idx = ws.max_row
        # color the whole row green/red
        fill = GREEN_FILL if correct else RED_FILL
        for c in range(1, 7):
            ws.cell(row=row_idx, column=c).fill = fill

    # Save CSV
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerows(csv_rows)

    # Save XLSX
    wb.save(OUTPUT_XLSX)

    # Summary
    print()
    print(YELLOW + "Summary:" + RESET)
    print(f"Total images processed: {total}")
    print(GREEN + f"Correct predictions: {correct_count}" + RESET)
    print(RED + f"Wrong predictions: {total - correct_count}" + RESET)
    print()
    print(YELLOW + f"Results saved to: {OUTPUT_CSV} and {OUTPUT_XLSX}" + RESET)



In [9]:
if __name__ == "__main__":
    main()

Loading model from 'cnn_model_4.h5'...


Using target image size: (224, 224)
[1/1945] normal_0001.png => True: normal | Pred: normal | Conf: 0.0039 | Correct
[2/1945] normal_0002.png => True: normal | Pred: normal | Conf: 0.0054 | Correct
[3/1945] normal_0003.jpg => True: normal | Pred: normal | Conf: 0.0002 | Correct
[4/1945] normal_0004.jpg => True: normal | Pred: normal | Conf: 0.0002 | Correct
[5/1945] normal_0005.jpg => True: normal | Pred: normal | Conf: 0.0144 | Correct
[6/1945] normal_0006.jpeg => True: normal | Pred: normal | Conf: 0.0443 | Correct
[7/1945] normal_0007.jpeg => True: normal | Pred: osteoporosis | Conf: 0.9651 | Wrong
[8/1945] normal_0008.jpg => True: normal | Pred: normal | Conf: 0.0855 | Correct
[9/1945] normal_0009.jpg => True: normal | Pred: normal | Conf: 0.0004 | Correct
[10/1945] normal_0010.jpg => True: normal | Pred: normal | Conf: 0.0487 | Correct
[11/1945] normal_0011.jpg => True: normal | Pred: normal | Conf: 0.0023 | Correct
[12/1945] normal_0012.jpg => True: normal | Pred: osteoporosis | 

In [10]:
def decide_label_and_confidence(pred):
    """
    Accepts model prediction array and returns (predicted_label_str, confidence_float).
    Supports:
      - Sigmoid single output: shape (1,1) -> confidence = pred[0][0], label = osteoporosis if >= 0.5
      - Softmax two outputs: shape (1,2) -> confidence = prob of class 1, label accordingly
      - Other shapes: take last dim index of max
    Convention: class 0 -> 'normal', class 1 -> 'osteoporosis'
    """
    arr = np.array(pred)
    if arr.ndim == 0:
        # scalar
        conf = float(arr)
        label = "osteoporosis" if conf >= 0.45 else "normal"
        return label, conf
    # If shape (1,) or (1,1)
    if arr.size == 1:
        conf = float(arr.flatten()[0])
        label = "osteoporosis" if conf >= 0.45 else "normal"
        return label, conf
    # If two or more outputs
    # get probs for class 1 if exists
    flat = arr.reshape(-1, arr.shape[-1]) if arr.ndim > 1 else arr
    probs = flat[0]
    # if two-class softmax
    if probs.shape[0] == 2:
        conf = float(probs[1])
        label = "osteoporosis" if np.argmax(probs) == 1 else "normal"
        return label, conf
    # otherwise pick argmax as predicted class index
    idx = int(np.argmax(probs))
    # if we assume index 1 == osteoporosis
    conf = float(probs[idx])
    label = "osteoporosis" if idx == 1 else "normal"
    return label, conf

In [11]:
if __name__ == "__main__":
    main()

Loading model from 'cnn_model_4.h5'...


Using target image size: (224, 224)
[1/1945] normal_0001.png => True: normal | Pred: normal | Conf: 0.0039 | Correct
[2/1945] normal_0002.png => True: normal | Pred: normal | Conf: 0.0054 | Correct
[3/1945] normal_0003.jpg => True: normal | Pred: normal | Conf: 0.0002 | Correct
[4/1945] normal_0004.jpg => True: normal | Pred: normal | Conf: 0.0002 | Correct
[5/1945] normal_0005.jpg => True: normal | Pred: normal | Conf: 0.0144 | Correct
[6/1945] normal_0006.jpeg => True: normal | Pred: normal | Conf: 0.0443 | Correct
[7/1945] normal_0007.jpeg => True: normal | Pred: osteoporosis | Conf: 0.9651 | Wrong
[8/1945] normal_0008.jpg => True: normal | Pred: normal | Conf: 0.0855 | Correct
[9/1945] normal_0009.jpg => True: normal | Pred: normal | Conf: 0.0004 | Correct
[10/1945] normal_0010.jpg => True: normal | Pred: normal | Conf: 0.0487 | Correct
[11/1945] normal_0011.jpg => True: normal | Pred: normal | Conf: 0.0023 | Correct
[12/1945] normal_0012.jpg => True: normal | Pred: osteoporosis | 